In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

In [2]:
from google.colab import files
uploaded = files.upload()

Saving spam.csv to spam.csv


In [3]:



spam = pd.read_csv('spam.csv', encoding='latin-1')
spam = spam[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'messages'})
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(spam['messages']).toarray()

le = LabelEncoder()
y = le.fit_transform(spam['label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report

In [5]:
max_len = 50 # نحدد طول موحد لجميع الرسائل
X_train_padded = pad_sequences(X_train, maxlen=max_len, padding='post')
X_test_padded = pad_sequences(X_test, maxlen=max_len, padding='post')

In [6]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [8]:
print("--- بدء تدريب الـ Neural Network (Phase 3) ---")
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))


--- بدء تدريب الـ Neural Network (Phase 3) ---
Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.8934 - loss: 0.2861 - val_accuracy: 0.9713 - val_loss: 0.1143
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.9872 - loss: 0.0504 - val_accuracy: 0.9767 - val_loss: 0.0683
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9948 - loss: 0.0169 - val_accuracy: 0.9776 - val_loss: 0.0781
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9980 - loss: 0.0068 - val_accuracy: 0.9767 - val_loss: 0.0884
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9993 - loss: 0.0030 - val_accuracy: 0.9794 - val_loss: 0.0960


In [9]:
# تقييم الموديل
loss, accuracy = model.evaluate(X_test, y_test)

print(f"\nDeep Learning Model Accuracy on Test Data: {accuracy:.2%}")

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9794 - loss: 0.0960

Deep Learning Model Accuracy on Test Data: 97.94%


In [10]:
# model.summary(show_trainable=True)

In [24]:

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       965
           1       0.96      0.88      0.92       150

    accuracy                           0.98      1115
   macro avg       0.97      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [22]:

from transformers import pipeline

sample_data = spam.sample(n=5, random_state=42)

classifier = pipeline(
    task="text-classification",
    model="mrm8488/bert-tiny-finetuned-sms-spam-detection"
)

for idx, row in sample_data.iterrows():

    prediction = classifier(row['messages'])[0]

    print(prediction)

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mrm8488/bert-tiny-finetuned-sms-spam-detection
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'label': 'LABEL_0', 'score': 0.7893965244293213}
{'label': 'LABEL_0', 'score': 0.9358083009719849}
{'label': 'LABEL_1', 'score': 0.9059011340141296}
{'label': 'LABEL_0', 'score': 0.9381788372993469}
{'label': 'LABEL_1', 'score': 0.9065071940422058}


In [23]:
import gradio as gr
from transformers import pipeline

clf = pipeline("text-classification", model="mrm8488/bert-tiny-finetuned-sms-spam-detection")

def predict(msg):
    if not msg.strip(): return "Please enter text."
    res = clf(msg)[0]
    lbl = "SPAM " if '1' in res['label'] else "HAM "
    return f"Result: {lbl} ({res['score']:.2%})"

gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=2, placeholder="Type message..."),
    outputs="text",
    title="SMS Spam Detection System ",
    description="Check messages for Spam vs Ham."
).launch(share=True)

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mrm8488/bert-tiny-finetuned-sms-spam-detection
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e35e5c51fcf07073ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
